# 2D Kolmogorov turbulence: interactive loss audit

On the periodic square, the reconstruction target is $(u,v,p)$ and the solver evolves vorticity $\omega=v_x-u_y$:

$$\mathbf u_t+\mathbf u\cdot\nabla\mathbf u=-\nabla p+Re^{-1}\Delta\mathbf u+\sin(4y)\,\mathbf e_x,\qquad \nabla\cdot\mathbf u=0.$$

Following the turbulent setting studied by [Chandler and Kerswell](https://arxiv.org/abs/1207.4682), we use the periodic $2\pi\times2\pi$ domain, $Re=40$, and forcing mode $n=4$. We perturb the unstable laminar state and discard a burn-in before recording. Pressure is recovered from its periodic Poisson equation and fixed to zero spatial mean.


In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

case_name = "navier_stokes"
channel_names = ("u", "v", "p")
smoke_mode = os.environ.get("PHYCOFLOW_VIZ_SMOKE", "0") == "1"
seed = 19
n = 16 if smoke_mode else 64
domain_length = 2.0 * np.pi
reynolds_number = 40.0
viscosity = 1.0 / reynolds_number
forcing_amplitude = 1.0
forcing_wavenumber = 4
dt = 0.005 if smoke_mode else 0.01
burn_in_time = 0.02 if smoke_mode else 20.0
record_time = 0.04 if smoke_mode else 10.0
save_every = 1 if smoke_mode else 5
rng = np.random.default_rng(seed)


In [2]:
x = np.linspace(0.0, domain_length, n, endpoint=False)
y = np.linspace(0.0, domain_length, n, endpoint=False)
xx, yy = np.meshgrid(x, y, indexing="xy")
dx = domain_length / n
wave = 2.0 * np.pi * np.fft.fftfreq(n, d=dx)
kx, ky = np.meshgrid(wave, wave, indexing="xy")
k2 = kx**2 + ky**2
inverse_k2 = np.zeros_like(k2)
inverse_k2[k2 > 0.0] = 1.0 / k2[k2 > 0.0]
modes = np.fft.fftfreq(n) * n
mx, my = np.meshgrid(modes, modes, indexing="xy")
dealias = (np.abs(mx) <= n / 3) & (np.abs(my) <= n / 3)
linear = -viscosity * k2

# Canonical Kolmogorov body force f=sin(n y)e_x is divergence-free.
force_potential = -forcing_amplitude * np.cos(forcing_wavenumber * yy) / forcing_wavenumber
force_x = forcing_amplitude * np.sin(forcing_wavenumber * yy)
force_y = np.zeros_like(force_x)
vorticity_forcing = -forcing_amplitude * forcing_wavenumber * np.cos(forcing_wavenumber * yy)
vorticity_forcing_hat = np.fft.fft2(vorticity_forcing)

def velocity_from_vorticity(omega_hat):
    psi_hat = inverse_k2 * omega_hat
    psi_hat[0, 0] = 0.0
    u = np.fft.ifft2(1j * ky * psi_hat).real
    v = np.fft.ifft2(-1j * kx * psi_hat).real
    return u, v

def nonlinear_term(omega_hat):
    u, v = velocity_from_vorticity(omega_hat)
    omega_x = np.fft.ifft2(1j * kx * omega_hat).real
    omega_y = np.fft.ifft2(1j * ky * omega_hat).real
    return (-np.fft.fft2(u * omega_x + v * omega_y) + vorticity_forcing_hat) * dealias

def smooth_initial_vorticity():
    raw_hat = np.fft.fft2(rng.standard_normal((n, n)))
    filt = np.exp(-0.025 * k2**2) * dealias
    perturbation = np.fft.ifft2(raw_hat * filt).real
    perturbation -= perturbation.mean()
    perturbation /= perturbation.std()
    laminar_vorticity = -(reynolds_number / forcing_wavenumber) * np.cos(forcing_wavenumber * yy)
    return laminar_vorticity + 0.5 * perturbation

contour_points = 16
roots = np.exp(1j * np.pi * (np.arange(1, contour_points + 1) - 0.5) / contour_points)
lr = dt * linear[..., None] + roots
E = np.exp(dt * linear)
E2 = np.exp(0.5 * dt * linear)
Q = dt * np.real(np.mean((np.exp(lr / 2.0) - 1.0) / lr, axis=-1))
f1 = dt * np.real(np.mean((-4.0 - lr + np.exp(lr) * (4.0 - 3.0 * lr + lr**2)) / lr**3, axis=-1))
f2 = dt * np.real(np.mean((2.0 + lr + np.exp(lr) * (-2.0 + lr)) / lr**3, axis=-1))
f3 = dt * np.real(np.mean((-4.0 - 3.0 * lr - lr**2 + np.exp(lr) * (4.0 - lr)) / lr**3, axis=-1))

omega_hat = np.fft.fft2(smooth_initial_vorticity()) * dealias
omega_snapshots, times_list = [], []
burn_steps = int(round(burn_in_time / dt))
record_steps = int(round(record_time / dt))
for step in range(burn_steps + record_steps + 1):
    if step >= burn_steps and (step - burn_steps) % save_every == 0:
        omega_snapshots.append(np.fft.ifft2(omega_hat).real)
        times_list.append((step - burn_steps) * dt)
    if step == burn_steps + record_steps:
        break
    Nv = nonlinear_term(omega_hat)
    a = E2 * omega_hat + Q * Nv
    Na = nonlinear_term(a)
    b = E2 * omega_hat + Q * Na
    Nb = nonlinear_term(b)
    c = E2 * a + Q * (2.0 * Nb - Nv)
    Nc = nonlinear_term(c)
    omega_hat = (E * omega_hat + f1 * Nv + 2.0 * f2 * (Na + Nb) + f3 * Nc) * dealias
    omega_hat[0, 0] = 0.0

times = np.asarray(times_list)
vorticity = np.asarray(omega_snapshots)


In [3]:
def spectral_derivative(field, direction):
    multiplier = 1j * (kx if direction == "x" else ky)
    return np.fft.ifft2(multiplier * np.fft.fft2(field)).real

def pressure_from_velocity(u, v):
    ux, uy = spectral_derivative(u, "x"), spectral_derivative(u, "y")
    vx, vy = spectral_derivative(v, "x"), spectral_derivative(v, "y")
    adv_u = np.fft.ifft2(np.fft.fft2(u * ux + v * uy) * dealias).real
    adv_v = np.fft.ifft2(np.fft.fft2(u * vx + v * vy) * dealias).real
    divergence_hat = 1j * kx * np.fft.fft2(adv_u) + 1j * ky * np.fft.fft2(adv_v)
    pressure_hat = divergence_hat * inverse_k2 * dealias
    pressure_hat[0, 0] = 0.0
    return np.fft.ifft2(pressure_hat).real

field_snapshots = []
for omega in vorticity:
    u, v = velocity_from_vorticity(np.fft.fft2(omega))
    p = pressure_from_velocity(u, v)
    field_snapshots.append(np.stack((u, v, p)))
fields = np.asarray(field_snapshots)  # [time, channel=(u,v,p), y, x]


In [4]:
def laplacian(field):
    return np.fft.ifft2(-k2 * np.fft.fft2(field)).real

divergence_rms = []
for u, v, _ in fields:
    divergence = spectral_derivative(u, "x") + spectral_derivative(v, "y")
    divergence_rms.append(np.sqrt(np.mean(divergence**2)))

momentum_u, momentum_v, scale_u, scale_v = [], [], [], []
for index in range(1, times.size - 1):
    u, v, p = fields[index]
    ut = (fields[index + 1, 0] - fields[index - 1, 0]) / (times[index + 1] - times[index - 1])
    vt = (fields[index + 1, 1] - fields[index - 1, 1]) / (times[index + 1] - times[index - 1])
    ux, uy = spectral_derivative(u, "x"), spectral_derivative(u, "y")
    vx, vy = spectral_derivative(v, "x"), spectral_derivative(v, "y")
    px, py = spectral_derivative(p, "x"), spectral_derivative(p, "y")
    adv_u = np.fft.ifft2(np.fft.fft2(u * ux + v * uy) * dealias).real
    adv_v = np.fft.ifft2(np.fft.fft2(u * vx + v * vy) * dealias).real
    ru = ut + adv_u + px - viscosity * laplacian(u) - force_x
    rv = vt + adv_v + py - viscosity * laplacian(v) - force_y
    momentum_u.append(ru); momentum_v.append(rv)
    scale_u.append(ut); scale_v.append(vt)
diagnostics = {
    "relative_momentum_residual_u": float(np.linalg.norm(momentum_u) / (np.linalg.norm(scale_u) + 1.0e-12)),
    "relative_momentum_residual_v": float(np.linalg.norm(momentum_v) / (np.linalg.norm(scale_v) + 1.0e-12)),
    "maximum_divergence_rms": float(np.max(divergence_rms)),
    "maximum_pressure_mean": float(np.max(np.abs(fields[:, 2].mean(axis=(-2, -1))))),
    "finite": bool(np.isfinite(fields).all()),
    "mean_enstrophy": float(np.mean(vorticity**2) / 2.0),
    "temporal_vorticity_std": float(vorticity.std(axis=0).mean()),
}
diagnostics


{'relative_momentum_residual_u': 0.005263550439625214,
 'relative_momentum_residual_v': 0.0046267532886192266,
 'maximum_divergence_rms': 4.771606591461598e-15,
 'maximum_pressure_mean': 4.163336342344337e-17,
 'finite': True,
 'mean_enstrophy': 4.150257679320837,
 'temporal_vorticity_std': 2.0881106623375167}

In [5]:
radius_index = np.sqrt(mx**2 + my**2).astype(int)

def kinetic_energy_spectrum(u, v):
    power = (np.abs(np.fft.fft2(u))**2 + np.abs(np.fft.fft2(v))**2) / (2.0 * u.size**2)
    total = np.bincount(radius_index.ravel(), weights=power.ravel())
    count = np.maximum(np.bincount(radius_index.ravel()), 1)
    return np.arange(total.size), total / count

def draw_frame(frame_index, axes):
    for axis in axes.ravel():
        axis.clear()
    u, v, p = fields[frame_index]
    omega = vorticity[frame_index]
    axes[0, 0].imshow(u, origin="lower", cmap="RdBu_r")
    axes[0, 0].set_title(f"u, t={times[frame_index]:.2f}")
    axes[0, 1].imshow(v, origin="lower", cmap="RdBu_r")
    axes[0, 1].set_title(f"v, t={times[frame_index]:.2f}")
    axes[0, 2].imshow(p, origin="lower", cmap="coolwarm")
    axes[0, 2].set_title("p (zero-mean gauge)")
    stride = max(1, u.size // 2000)
    cloud = axes[1, 0].scatter(u.ravel()[::stride], v.ravel()[::stride], c=p.ravel()[::stride], s=6, cmap="coolwarm", alpha=0.5)
    axes[1, 0].set(xlabel="u", ylabel="v", title="Joint (u,v,p) cloud; color=p")
    radius, energy = kinetic_energy_spectrum(u, v)
    axes[1, 1].loglog(radius[1:], energy[1:] + 1.0e-18, color="black")
    axes[1, 1].set_title("Kolmogorov-flow energy spectrum")
    levels = np.quantile(omega, [0.2, 0.4, 0.6, 0.8])
    span = max(float(np.ptp(omega)), 1.0e-12)
    bounds = np.r_[omega.min() - 1.0e-6 * span, levels, omega.max() + 1.0e-6 * span]
    axes[1, 2].contourf(omega, levels=bounds, cmap="Spectral_r")
    axes[1, 2].contour(omega, levels=levels, colors="black", linewidths=0.4)
    axes[1, 2].set_title("Vorticity level-set geometry")
    return [cloud]

dashboard = None
if not smoke_mode:
    frame_indices = np.unique(np.linspace(0, times.size - 1, min(times.size, 31), dtype=int))
    figure, dashboard_axes = plt.subplots(2, 3, figsize=(12, 7), dpi=72, constrained_layout=True)
    animation = FuncAnimation(figure, lambda i: draw_frame(i, dashboard_axes), frames=frame_indices, interval=180, repeat=True)
    plt.close(figure)
    dashboard = HTML(animation.to_jshtml(default_mode="once"))
    display(dashboard)
